### Transform drivers data
1. Read "bronze" drivers table
2. Keep ony columns for analitycs, drop column url
3. Standarize column name using snake_case (driverId -> driver_id, dateOfBirth -> date_of_birth)
4. Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case.
5. Remove duplicate records
7. Transform values of columns nationality to Title Case
8. Write transformed data to silver drivers table

##### 1. Read "bronze" drivers table

In [0]:
%run ../Workspace/common/configuration_environment

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)

In [0]:
drivers_df = spark.table(bronze_table)
display(drivers_df)

_corrupt_record,constructorId,dateOfBirth,driverId,name,nationality,url,ingestion_timestamp,source_file
null,null,1932-07-10,abate,"List(carlo, abate)",italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-03-21,abecassis,"List(george, abecassis)",british,http://en.wikipedia.org/wiki/George_Abecassis,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1957-11-27,acheson,"List(kenny, acheson)",british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1969-11-19,adams,"List(philippe, adams)",belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-12-15,ader,"List(walt, ader)",american,http://en.wikipedia.org/wiki/Walt_Ader,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1921-11-05,adolff,"List(kurt, adolff)",german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-08-21,agabashian,"List(fred, agabashian)",american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1940-04-19,ahrens,"List(kurt, ahrens)",german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1995-09-23,aitken,"List(jack, aitken)",british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1979-04-16,albers,"List(christijan, albers)",dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json


##### 2. Keep ony columns for analitycs, drop column url

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_selected_df = drivers_df.select(
    "dateOfBirth",
    "driverId",
    "name",
    "nationality",
    "url",
    "ingestion_timestamp",
    "source_file"
)

display(drivers_selected_df)

dateOfBirth,driverId,name,nationality,url,ingestion_timestamp,source_file
1932-07-10,abate,"List(carlo, abate)",italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1913-03-21,abecassis,"List(george, abecassis)",british,http://en.wikipedia.org/wiki/George_Abecassis,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1957-11-27,acheson,"List(kenny, acheson)",british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1969-11-19,adams,"List(philippe, adams)",belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1913-12-15,ader,"List(walt, ader)",american,http://en.wikipedia.org/wiki/Walt_Ader,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1921-11-05,adolff,"List(kurt, adolff)",german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1913-08-21,agabashian,"List(fred, agabashian)",american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1940-04-19,ahrens,"List(kurt, ahrens)",german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1995-09-23,aitken,"List(jack, aitken)",british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1979-04-16,albers,"List(christijan, albers)",dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json


In [0]:
drivers_dropped_df = drivers_selected_df.drop(F.col("url"))

##### 3. Standarize column name using snake_case (driverId -> driver_id, dateOfBirth -> date_of_birth)

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnsRenamed({"driverId": "driver_id",
                             "dateOfBirth": "date_of_birth" })
)

In [0]:
display(drivers_renamed_df)

date_of_birth,driver_id,name,nationality,ingestion_timestamp,source_file
1932-07-10,abate,"List(carlo, abate)",italian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1913-03-21,abecassis,"List(george, abecassis)",british,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1957-11-27,acheson,"List(kenny, acheson)",british,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1969-11-19,adams,"List(philippe, adams)",belgian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1913-12-15,ader,"List(walt, ader)",american,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1921-11-05,adolff,"List(kurt, adolff)",german,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1913-08-21,agabashian,"List(fred, agabashian)",american,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1940-04-19,ahrens,"List(kurt, ahrens)",german,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1995-09-23,aitken,"List(jack, aitken)",british,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json
1979-04-16,albers,"List(christijan, albers)",dutch,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json


##### 4. Concatenate name.givenName and name.familyName to create a new column called driver_name and transform the value to Title Case.
##### 6. Transform values of columns nationality to Title Case

In [0]:
drivers_concatenated_df = (
    drivers_renamed_df
        .withColumn("driver_name", F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName")))
)

display(drivers_concatenated_df)

date_of_birth,driver_id,name,nationality,ingestion_timestamp,source_file,driver_name
1932-07-10,abate,"List(carlo, abate)",italian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,carlo abate
1913-03-21,abecassis,"List(george, abecassis)",british,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,george abecassis
1957-11-27,acheson,"List(kenny, acheson)",british,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,kenny acheson
1969-11-19,adams,"List(philippe, adams)",belgian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,philippe adams
1913-12-15,ader,"List(walt, ader)",american,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,walt ader
1921-11-05,adolff,"List(kurt, adolff)",german,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,kurt adolff
1913-08-21,agabashian,"List(fred, agabashian)",american,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,fred agabashian
1940-04-19,ahrens,"List(kurt, ahrens)",german,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,kurt ahrens
1995-09-23,aitken,"List(jack, aitken)",british,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,jack aitken
1979-04-16,albers,"List(christijan, albers)",dutch,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,christijan albers


In [0]:
drivers_distinct_df = (
    drivers_concatenated_df
    .withColumn('nationality', F.initcap(F.col('nationality')))
    .withColumn('driver_name', F.initcap(F.col('driver_name')))
)

display(drivers_distinct_df)

date_of_birth,driver_id,name,nationality,ingestion_timestamp,source_file,driver_name
1932-07-10,abate,"List(carlo, abate)",Italian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Carlo Abate
1913-03-21,abecassis,"List(george, abecassis)",British,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,George Abecassis
1957-11-27,acheson,"List(kenny, acheson)",British,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kenny Acheson
1969-11-19,adams,"List(philippe, adams)",Belgian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Philippe Adams
1913-12-15,ader,"List(walt, ader)",American,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Walt Ader
1921-11-05,adolff,"List(kurt, adolff)",German,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Adolff
1913-08-21,agabashian,"List(fred, agabashian)",American,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Fred Agabashian
1940-04-19,ahrens,"List(kurt, ahrens)",German,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Ahrens
1995-09-23,aitken,"List(jack, aitken)",British,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jack Aitken
1979-04-16,albers,"List(christijan, albers)",Dutch,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Christijan Albers


##### 5. Remove duplicate records

In [0]:
drivers_final_df = drivers_distinct_df.dropDuplicates(["driver_id"])
display(drivers_final_df)

date_of_birth,driver_id,name,nationality,ingestion_timestamp,source_file,driver_name
1932-07-10,abate,"List(carlo, abate)",Italian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Carlo Abate
1913-03-21,abecassis,"List(george, abecassis)",British,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,George Abecassis
1957-11-27,acheson,"List(kenny, acheson)",British,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kenny Acheson
1969-11-19,adams,"List(philippe, adams)",Belgian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Philippe Adams
1913-12-15,ader,"List(walt, ader)",American,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Walt Ader
1921-11-05,adolff,"List(kurt, adolff)",German,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Adolff
1913-08-21,agabashian,"List(fred, agabashian)",American,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Fred Agabashian
1940-04-19,ahrens,"List(kurt, ahrens)",German,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Ahrens
1995-09-23,aitken,"List(jack, aitken)",British,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jack Aitken
1979-04-16,albers,"List(christijan, albers)",Dutch,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Christijan Albers


##### 7. Write transformed data to silver circuits table

In [0]:
(
    drivers_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)    
)

In [0]:
display(
    spark.read.table(silver_table)
)

date_of_birth,driver_id,name,nationality,ingestion_timestamp,source_file,driver_name
1940-04-19,ahrens,"List(kurt, ahrens)",German,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Ahrens
1961-04-20,barilla,"List(paolo, barilla)",Italian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Paolo Barilla
1914-02-28,bayol,"List(élie, bayol)",French,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Élie Bayol
1924-01-07,birger,"List(pablo, birger)",Argentine,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Pablo Birger
1946-11-25,borgudd,"List(slim, borgudd)",Swedish,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Slim Borgudd
1916-09-15,branca,"List(toni, branca)",Swiss,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Toni Branca
1949-12-24,brown,"List(warwick, brown)",Australian,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Warwick Brown
1924-04-04,christie,"List(bob, christie)",American,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Bob Christie
1936-03-04,clark,"List(jim, clark)",British,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jim Clark
1906-08-16,george_connor,"List(george, connor)",American,2026-04-27T13:15:15.144Z,dbfs:/Volumes/formula1/landing/files/drivers.json,George Connor
